In [ ]:
from pathlib import Path

def list_rst_paths(root_dir):
    root = Path(root_dir)
    if not root.is_dir():
        raise FileNotFoundError(f"Source directory not found: {root}")
    
    paths = []
    # subdirs = ["", "tabular-data", "statistical-modeling"]
    subdirs = ["", "combinators", "context", "measurements", "plugins", 
                "polars", "polars/expressions", "programming-framework", 
                "transformations", "utilities"]
    for sub in subdirs:
        sub_path = root / sub
        if sub_path.is_dir():
            # Use glob("*.rst") to get only direct children, not recursive
            paths.extend(sub_path.glob("*.rst"))
    
    return sorted(paths)

def save_rst_as_txt(source_root, target_root):
    source_root = Path(source_root)
    target_root = Path(target_root)

    if not source_root.is_dir():
        raise FileNotFoundError(f"Source directory not found: {source_root}")

    for rst_path in list_rst_paths(source_root):
        rel_path = rst_path.relative_to(source_root).with_suffix(".txt")
        out_path = target_root / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(rst_path.read_text(encoding="utf-8"), encoding="utf-8")
        print(f"Saved {rst_path} -> {out_path}")

In [23]:
save_rst_as_txt(source_root="opendp/docs/source/api/user-guide", target_root="rag_corpus/user-guide-api")

Saved opendp/docs/source/api/user-guide/combinators/adaptive-composition.rst -> rag_corpus/user-guide-api/combinators/adaptive-composition.txt
Saved opendp/docs/source/api/user-guide/combinators/amplification.rst -> rag_corpus/user-guide-api/combinators/amplification.txt
Saved opendp/docs/source/api/user-guide/combinators/chaining.rst -> rag_corpus/user-guide-api/combinators/chaining.txt
Saved opendp/docs/source/api/user-guide/combinators/compositor-chaining-and-nesting.rst -> rag_corpus/user-guide-api/combinators/compositor-chaining-and-nesting.txt
Saved opendp/docs/source/api/user-guide/combinators/fully-adaptive-composition.rst -> rag_corpus/user-guide-api/combinators/fully-adaptive-composition.txt
Saved opendp/docs/source/api/user-guide/combinators/index.rst -> rag_corpus/user-guide-api/combinators/index.txt
Saved opendp/docs/source/api/user-guide/combinators/measure-casting.rst -> rag_corpus/user-guide-api/combinators/measure-casting.txt
Saved opendp/docs/source/api/user-guide/com

In [27]:
from pathlib import Path
import re

literalinclude_re = re.compile(r"^\s*\.\.\s+literalinclude::\s+(?P<target>\S+)\s*$")
option_re = re.compile(r"^\s*:(?P<key>[\w-]+):\s*(?P<value>.*)\s*$")

def extract_literalinclude_blocks(text: str):
    lines = text.splitlines()
    i = 0
    blocks = []

    while i < len(lines):
        m = literalinclude_re.match(lines[i])
        if not m:
            i += 1
            continue

        target = m.group("target")
        indent = len(lines[i]) - len(lines[i].lstrip())

        options = {}
        j = i + 1

        while j < len(lines):
            line = lines[j]

            # stop when indentation drops or we hit a new directive/paragraph
            current_indent = len(line) - len(line.lstrip())
            if line.strip() == "":
                j += 1
                continue
            if current_indent <= indent:
                break

            om = option_re.match(line)
            if om:
                key = om.group("key")
                value = om.group("value").strip()
                options[key] = value if value else True

            j += 1

        blocks.append({
            "target": target,
            "start-after": options.get("start-after"),
            "end-before": options.get("end-before"),
            "language": options.get("language"),
            "dedent": "dedent" in options,
            "start_line": i,
            "end_line": j
        })

        i = j

    return blocks

def extract_code_section(file_path, start_after, end_before):
    lines = file_path.read_text(encoding='utf-8').splitlines()
    
    start_idx = 0
    if start_after:
        for i, line in enumerate(lines):
            if start_after in line:
                start_idx = i + 1
                break
    
    end_idx = len(lines)
    if end_before:
        for i, line in enumerate(lines[start_idx:], start_idx):
            if end_before in line:
                end_idx = i
                break
    
    return '\n'.join(lines[start_idx:end_idx])

def print_blocks(blocks):
    for idx, b in enumerate(blocks, 1):
        print(f"Block {idx}")
        print(f"  target      : {b['target']}")
        print(f"  start-after : {b['start-after']}")
        print(f"  end-before  : {b['end-before']}")
        print(f"  language    : {b['language']}")
        print(f"  dedent      : {b['dedent']}")
        print(f"  start_line  : {b['start_line']}")
        print(f"  end_line    : {b['end_line']}")
        print()

def process_rst_with_code(text, source_root, output_file=None):
    blocks = extract_literalinclude_blocks(text)
    print_blocks(blocks)
    # Sort blocks by start_line in descending order to replace from the end
    blocks.sort(key=lambda b: b['start_line'], reverse=True)
    
    lines = text.splitlines()
    for block in blocks:
        if block['language'] == 'python':
            target_path = Path(source_root) / block['target']
            if target_path.exists():
                code = extract_code_section(target_path, block['start-after'], block['end-before'])
                # Add the .. code:: pycon directive before the code
                replacement = ".. code:: pycon\n\n" + code
                # Indent each line with 2 tabs and add 2 new lines at the end
                indented_replacement = "\n".join("\t\t" + line for line in replacement.splitlines()) + "\n\n"
                # Replace the block lines with the indented replacement lines
                lines = lines[:block['start_line']] + indented_replacement.splitlines() + lines[block['end_line']:]
    
    # return '\n'.join(lines)
    result = '\n'.join(lines)
    if output_file:
        Path(output_file).write_text(result, encoding='utf-8')
    return result

In [30]:
fn = "rag_corpus/user-guide-api/plugins/transformation.txt"

FILE = Path(fn)
text = FILE.read_text(encoding="utf-8")
source_root = 'opendp/docs/source/api/user-guide/plugins'

process_rst_with_code(text, source_root, fn)

Block 1
  target      : code/transformation.rst
  start-after : # enable-features
  end-before  : # /enable-features
  language    : python
  dedent      : True
  start_line  : 15
  end_line    : 21

Block 2
  target      : code/transformation.R
  start-after : library(opendp)
  end-before  : # make-repeat
  language    : r
  dedent      : False
  start_line  : 24
  end_line    : 29

Block 3
  target      : code/transformation.rst
  start-after : # make-repeat
  end-before  : # /make-repeat
  language    : python
  dedent      : True
  start_line  : 36
  end_line    : 42

Block 4
  target      : code/transformation.R
  start-after : # make-repeat
  end-before  : # /make-repeat
  language    : r
  dedent      : False
  start_line  : 45
  end_line    : 50

Block 5
  target      : code/transformation.rst
  start-after : # use-transformation
  end-before  : # /use-transformation
  language    : python
  dedent      : True
  start_line  : 57
  end_line    : 63

Block 6
  target      : code/

'\nTransformation example\n======================\n\nUse :func:`~opendp.transformations.make_user_transformation` to construct your own transformation.\n\n.. note::\n\n    This requires a looser trust model, as we cannot verify any privacy or stability properties of user-defined functions.\n\n    .. tab-set::\n\n        .. tab-item:: Python\n            :sync: python\n\n\t\t.. code:: pycon\n\t\t\n\t\t    >>> import opendp.prelude as dp\n\t\t    >>> dp.enable_features("honest-but-curious", "contrib")\n\n        .. tab-item:: R\n            :sync: r\n\n            .. literalinclude:: code/transformation.R\n                :language: r\n                :start-after: library(opendp)\n                :end-before: # make-repeat\n\nIn this example, we mock the typical API of the OpenDP library to make a transformation that duplicates each record ``multiplicity`` times:\n\n.. tab-set::\n\n    .. tab-item:: Python\n        :sync: python\n\n\t\t.. code:: pycon\n\t\t\n\t\t    >>> def make_repeat(

In [31]:
import json
from pathlib import Path

def print_notebook_readable(notebook_path):
    notebook_path = Path(notebook_path)
    
    if not notebook_path.exists():
        raise FileNotFoundError(f"Notebook not found: {notebook_path}")
    
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)
    
    # Print metadata
    print("=" * 80)
    print(f"Notebook: {notebook_path.name}")
    print("=" * 80)
    
    cells = notebook.get('cells', [])
    
    for idx, cell in enumerate(cells, 1):
        cell_type = cell.get('cell_type', 'unknown')
        source = cell.get('source', [])
        
        # Join source lines if it's a list
        if isinstance(source, list):
            source_text = ''.join(source)
        else:
            source_text = source
        
        print(f"\n{'─' * 80}")
        print(f"Cell {idx} ({cell_type.upper()})")
        print(f"{'─' * 80}")
        print(source_text)
    
    print(f"\n{'=' * 80}")
    print(f"Total cells: {len(cells)}")
    print(f"{'=' * 80}")

# Usage:
print_notebook_readable("opendp/docs/source/api/user-guide/measurements/additive-noise-mechanisms.ipynb")

Notebook: additive-noise-mechanisms.ipynb

────────────────────────────────────────────────────────────────────────────────
Cell 1 (MARKDOWN)
────────────────────────────────────────────────────────────────────────────────
# Additive Noise Mechanisms

This notebook documents the variations on additive noise mechanisms in OpenDP:

* Distribution: Laplace vs. Gaussian
* Support: float vs. integer
* Domain: scalar vs. vector
* Bit-depth

────────────────────────────────────────────────────────────────────────────────
Cell 2 (MARKDOWN)
────────────────────────────────────────────────────────────────────────────────
----
Any constructors that have not completed the proof-writing and vetting process may still be accessed if you opt-in to "contrib".
Please contact us if you are interested in proof-writing. Thank you!

────────────────────────────────────────────────────────────────────────────────
Cell 3 (CODE)
────────────────────────────────────────────────────────────────────────────────
i

In [33]:
import json
from pathlib import Path

def list_ipynb_paths(root_dir):
    """List only .ipynb files from subdirectories"""
    root = Path(root_dir)
    if not root.is_dir():
        raise FileNotFoundError(f"Source directory not found: {root}")
    
    paths = []
    subdirs = ["", "combinators", "context", "measurements", "plugins", 
                "polars", "polars/expressions", "programming-framework", 
                "transformations", "utilities"]
    for sub in subdirs:
        sub_path = root / sub
        if sub_path.is_dir():
            paths.extend(sub_path.glob("*.ipynb"))
    
    return sorted(paths)

def notebook_to_readable_text(notebook_path):
    """Convert notebook to readable text format optimized for RAG"""
    notebook_path = Path(notebook_path)
    
    if not notebook_path.exists():
        raise FileNotFoundError(f"Notebook not found: {notebook_path}")
    
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)
    
    output = []
    cells = notebook.get('cells', [])
    
    for idx, cell in enumerate(cells, 1):
        cell_type = cell.get('cell_type', 'unknown')
        source = cell.get('source', [])
        
        # Join source lines if it's a list
        if isinstance(source, list):
            source_text = ''.join(source).strip()
        else:
            source_text = source.strip()
        
        if not source_text:
            continue
        
        # Use markdown headers instead of decorative lines
        if cell_type == 'markdown':
            output.append(source_text)
        else:  # code cell
            output.append(f"```python\n{source_text}\n```")
        
        output.append("")  # blank line for spacing
    
    return '\n'.join(output)

def save_ipynb_as_txt(source_root, target_root):
    """Convert and save .ipynb files as readable text files for RAG"""
    source_root = Path(source_root)
    target_root = Path(target_root)

    if not source_root.is_dir():
        raise FileNotFoundError(f"Source directory not found: {source_root}")

    for ipynb_path in list_ipynb_paths(source_root):
        rel_path = ipynb_path.relative_to(source_root).with_suffix(".txt")
        out_path = target_root / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        readable_text = notebook_to_readable_text(ipynb_path)
        out_path.write_text(readable_text, encoding='utf-8')
        print(f"Converted {ipynb_path} -> {out_path}")

In [34]:
save_ipynb_as_txt(source_root="opendp/docs/source/api/user-guide", target_root="rag_corpus/user-guide-api")

Converted opendp/docs/source/api/user-guide/measurements/additive-noise-mechanisms.ipynb -> rag_corpus/user-guide-api/measurements/additive-noise-mechanisms.txt
Converted opendp/docs/source/api/user-guide/measurements/canonical-noise-mechanism.ipynb -> rag_corpus/user-guide-api/measurements/canonical-noise-mechanism.txt
Converted opendp/docs/source/api/user-guide/polars/data-types.ipynb -> rag_corpus/user-guide-api/polars/data-types.txt
Converted opendp/docs/source/api/user-guide/polars/expressions/aggregation.ipynb -> rag_corpus/user-guide-api/polars/expressions/aggregation.txt
Converted opendp/docs/source/api/user-guide/polars/expressions/columns.ipynb -> rag_corpus/user-guide-api/polars/expressions/columns.txt
Converted opendp/docs/source/api/user-guide/polars/expressions/manipulation.ipynb -> rag_corpus/user-guide-api/polars/expressions/manipulation.txt
Converted opendp/docs/source/api/user-guide/polars/expressions/operators.ipynb -> rag_corpus/user-guide-api/polars/expressions/ope

In [ ]:
import os
import re
from pathlib import Path


def extract_rst_paths(root_dir):
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"Directory not found: {root_dir}")

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in sorted(filenames):
            if filename.endswith(".rst"):
                print(os.path.join(dirpath, filename))

In [ ]:
extract_rst_paths("opendp/docs/source/getting-started")

opendp/docs/source/getting-started/index.rst
opendp/docs/source/getting-started/next-steps.rst
opendp/docs/source/getting-started/quickstart.rst
opendp/docs/source/getting-started/typical-workflow.rst
opendp/docs/source/getting-started/utility.rst
opendp/docs/source/getting-started/statistical-modeling/index.rst
opendp/docs/source/getting-started/statistical-modeling/linear-regression.rst
opendp/docs/source/getting-started/statistical-modeling/pca.rst
opendp/docs/source/getting-started/statistical-modeling/synthetic-data.rst
opendp/docs/source/getting-started/tabular-data/contingency-tables.rst
opendp/docs/source/getting-started/tabular-data/essential-statistics.rst
opendp/docs/source/getting-started/tabular-data/identifier-truncation.rst
opendp/docs/source/getting-started/tabular-data/index.rst
opendp/docs/source/getting-started/tabular-data/preparing-microdata.rst
opendp/docs/source/getting-started/code/quickstart-context.rst
opendp/docs/source/getting-started/code/typical-workflow-c